In [1]:
import os
os.chdir("C:/journal_project")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [2]:
df = pd.read_csv("outputs/unified_features.csv")

FEATURE_SETS = {
    "Original_7_features": ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"],
    "Expanded_9_features":  ["hr", "hrv", "sdnn", "pnn50", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"],
}

le = LabelEncoder()
y = le.fit_transform(df["ground_truth"])
groups = df["subject"].values

In [3]:
N_REPEATS = 10
results = {}
for name, feats in FEATURE_SETS.items():
    df_clean = df.dropna(subset=feats)
    X = df_clean[feats].values
    y_clean = le.transform(df_clean["ground_truth"])
    groups_clean = df_clean["subject"].values
    fold_f1 = []
    for repeat in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for train_idx, test_idx in sgkf.split(X, y_clean, groups=groups_clean):
            scaler = StandardScaler()
            Xtr, Xte = scaler.fit_transform(X[train_idx]), scaler.transform(X[test_idx])
            clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
            clf.fit(Xtr, y_clean[train_idx])
            fold_f1.append(f1_score(y_clean[test_idx], clf.predict(Xte), average="macro", zero_division=0))
    results[name] = fold_f1
    print(f"{name}: F1 = {np.mean(fold_f1):.4f} (SD={np.std(fold_f1):.4f})")

from scipy import stats
t, p = stats.wilcoxon(results["Expanded_9_features"], results["Original_7_features"])
print(f"\nExpanded vs Original: p={p:.4f}")

Original_7_features: F1 = 0.7306 (SD=0.0912)
Expanded_9_features: F1 = 0.7412 (SD=0.0956)

Expanded vs Original: p=0.3177
